# Train the complexity router on a Colab GPU

Trains a **pairwise tier scorer** on `router_training_pairs.jsonl`: the model reads
*(prompt, complexity tier)* and predicts whether that tier **wins (1.0), ties (0.5) or
loses (0.0)** on the prompt. Each row of the file is exactly `{prompt, tier, outcome, source}` —
the trainer consumes **only `prompt`, `tier` and `outcome`**; the `source` column is ignored.

Routing = score the prompt against the five tier descriptions and pick the argmax.

A free T4 is plenty: 15k pairs x 3 epochs is ~1,400 steps, roughly 10 minutes.

**Before you start:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. Build the pairs file locally (it is gitignored, so Colab can't clone it):
   ```
   python scripts/build_training_pairs.py --max-pairs 15000
   ```
   Drop `--max-pairs` for the full ~534k pairs once the pipeline is proven (training time
   scales ~35x).

In [ ]:
# Should list an NVIDIA GPU. If not: Runtime > Change runtime type > GPU, then re-run.
!nvidia-smi

## 1. Upload the training pairs

In [ ]:
import os, shutil
from google.colab import files

DATA_PATH = "/content/data/router_training_pairs.jsonl"
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
if not os.path.exists(DATA_PATH):
    print("pick router_training_pairs.jsonl")
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, DATA_PATH)
print(DATA_PATH, os.path.getsize(DATA_PATH) // 1024, "KB")
# Already on Drive? Skip the upload and set:
# DATA_PATH = "/content/drive/MyDrive/llm-router/router_training_pairs.jsonl"

## 2. Dependencies

Colab's preinstalled CUDA torch stays untouched — only `transformers` is needed.

In [ ]:
!pip install -q transformers

## 3. Output location (Drive recommended)

Free Colab VMs are wiped when the session ends — save straight to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/llm-router/complexity_router"
# Prefer the ephemeral VM instead? Use:
# OUTPUT_DIR = "/content/complexity_router"

## 4. Data prep — `prompt`, `tier`, `outcome` only

The validation split is **prompt-grouped** (a stable hash of the prompt), so no prompt
appears in both train and val.

In [ ]:
import collections
import hashlib
import json

TIERS = (1, 2, 3, 4, 5)
TIER_TEXTS = {
    1: "complexity tier 1 - a tiny flash-tier model for simple tasks",
    2: "complexity tier 2 - a light mini-tier model",
    3: "complexity tier 3 - a mid-tier pro model",
    4: "complexity tier 4 - a high-capability frontier model",
    5: "complexity tier 5 - a top frontier model for the hardest tasks",
}
MAX_LEN = 256        # prompts are truncated; enough signal, keeps the T4 fast
VAL_PROMPT_MOD = 20  # every 20th unique prompt (by hash) -> validation (~5%)

rows = []
with open(DATA_PATH) as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        # only these three fields are used; `source` is deliberately ignored
        rows.append((r["prompt"], int(r["tier"]), float(r["outcome"])))

train, val = [], []
for triplet in rows:
    prompt = triplet[0]
    is_val = int(hashlib.sha1(prompt.encode()).hexdigest(), 16) % VAL_PROMPT_MOD == 0
    (val if is_val else train).append(triplet)

print(f"pairs: {len(rows)} (train {len(train)} / val {len(val)})")
print("outcome mix:", dict(sorted(collections.Counter(o for _p, _t, o in rows).items())))
print("tier mix:  ", dict(sorted(collections.Counter(t for _p, t, _o in rows).items())))

## 5. Model + loaders

Same backbone family the repo always used (a 6-layer MiniLM cross-encoder, ~22M params),
fed `prompt` + a fixed name-free description of the tier. The manual PyTorch loop (no HF
Trainer, no `pin_memory`) keeps every accelerator working: cuda -> mps -> directml -> cpu.

In [ ]:
import importlib

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    try:
        torch_directml = importlib.import_module("torch_directml")  # never a hard import
        return torch_directml.device()
    except (ImportError, OSError):
        return torch.device("cpu")

DEVICE = pick_device()
print("device:", DEVICE)

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEVICE)

class PairSet(Dataset):
    def __init__(self, triplets):
        self.items = triplets

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        prompt, tier, outcome = self.items[i]
        enc = tokenizer(prompt, TIER_TEXTS[tier], truncation=True, max_length=MAX_LEN)
        return enc["input_ids"], enc["attention_mask"], outcome

def collate(batch):
    input_ids, attention_mask, labels = zip(*batch)
    padded = tokenizer.pad({"input_ids": list(input_ids),
                            "attention_mask": list(attention_mask)},
                           return_tensors="pt")
    return padded, torch.tensor(labels, dtype=torch.float)

train_loader = DataLoader(PairSet(train), batch_size=32, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(PairSet(val), batch_size=64, collate_fn=collate)

## 6. Train

BCE on the raw logit with 0.5 targets for ties means the loss floor sits well below the
old 0.69 coin-flip line — watch **routing accuracy** (argmax over the five tier scores on
held-out prompts) as the real signal.

In [ ]:
EPOCHS = 3
LR = 2e-5
WARMUP_FRAC = 0.06

loss_fn = torch.nn.BCEWithLogitsLoss()
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps),
                                            total_steps)

@torch.no_grad()
def model_loss(loader) -> float:
    model.eval()
    total, n = 0.0, 0
    for padded, labels in loader:
        logits = model(**padded.to(DEVICE)).logits.squeeze(-1).cpu()
        total += loss_fn(logits, labels).item() * len(labels)
        n += len(labels)
    return total / max(n, 1)

@torch.no_grad()
def score(prompt: str, tier: int) -> float:
    was_training = model.training
    model.eval()
    enc = tokenizer(prompt, TIER_TEXTS[tier], truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    value = model(**enc).logits.squeeze(-1).item()
    if was_training:
        model.train()
    return value

@torch.no_grad()
def routing_accuracy(triplets) -> float:
    """On held-out prompts with a decisive battle (exactly one 1.0 outcome),
    does the argmax over the five tier scores pick the winning tier?"""
    by_prompt = {}
    for prompt, tier, outcome in triplets:
        by_prompt.setdefault(prompt, {})[tier] = outcome
    correct = total = 0
    for prompt, tier_out in by_prompt.items():
        gold = [t for t, o in tier_out.items() if o == 1.0]
        if len(gold) != 1:
            continue
        scores = {t: score(prompt, t) for t in TIERS}
        total += 1
        correct += max(scores, key=scores.get) == gold[0]
    return correct / max(total, 1)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for padded, labels in train_loader:
        optimizer.zero_grad()
        logits = model(**padded.to(DEVICE)).logits.squeeze(-1)
        loss = loss_fn(logits, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()
    print(f"epoch {epoch}/{EPOCHS} | train BCE {running / steps_per_epoch:.4f} | "
          f"val BCE {model_loss(val_loader):.4f} | "
          f"routing acc {routing_accuracy(val):.3f}")

## 7. Try it — route a prompt

In [ ]:
def route(prompt: str) -> None:
    probs = {t: torch.sigmoid(torch.tensor(score(prompt, t))).item() for t in TIERS}
    ranked = sorted(probs.items(), key=lambda kv: -kv[1])
    print(f"> {prompt[:100]}")
    for tier, p in ranked:
        arrow = "  <-- route here" if tier == ranked[0][0] else ""
        print(f"  tier {tier}: {p:.3f}{arrow}")
    print()

route("write a quicksort in python")
route("hey, what's a good name for my cat?")
route("prove that there are infinitely many prime numbers")

## 8. Save

The tier descriptions are part of the model contract — `tier_texts.json` is saved next to
the weights so inference uses exactly the strings training saw.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(f"{OUTPUT_DIR}/tier_texts.json", "w") as f:
    json.dump({str(k): v for k, v in TIER_TEXTS.items()}, f, indent=2)
print("saved to", OUTPUT_DIR)

## 9. Get the model back (skip if you trained to Drive)

Unzip into `models/complexity_router/` in your local repo.

In [ ]:
# from google.colab import files
# !zip -r /content/complexity_router.zip "{OUTPUT_DIR}"
# files.download("/content/complexity_router.zip")